In [2]:
import pandas as pd

# LOAD Labeled_Data_for_Exclusion_Included

In [5]:
processed_df = pd.read_csv("/content/Labeled_Data_for_Exclusion_Included_fixed_1.csv")
processed_df

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english,Decision
0,16985399.0,10.1097/00001786-200610000-00006,Missed nursing care: a qualitative study,2006,The purpose of this study was to determine nur...,[],en,Excluded\n\nExplain: The title and abstract of...
1,15946170.0,10.1111/j.1365-2934.2005.00541.x,Confirming mentorship,2005,Mentorship is related to nurses' success in nu...,[],en,**Decision:** Excluded\n\n**Explanation:** \n\...
2,17147757.0,10.1111/j.1741-6787.2004.04008.x,Methods in translation science,2004,The gap between research findings and practice...,[],en,**Decision:** Excluded\n\n**Explanation:** \nT...
3,19076465.0,10.1111/j.1744-6198.2008.00115.x,Competency: a concept analysis,2008,Concept analysis of competency.\nThe purpose o...,[],en,Decision: excluded\n\nExplain: The abstract in...
4,15946274.0,10.1111/j.1365-2702.2005.01136.x,Clinical education facilitators: a literature ...,2005,"The aim of this literature review, set within ...",[],en,Excluded\n\nExplain: The abstract of the paper...
...,...,...,...,...,...,...,...,...
15938,NaN,NaN,Enhancing Nursing Practice in Developmental Ca...,2008,This study aimed to enhance nursing practice i...,developmental care; participatory action resea...,en,**Decision:** Excluded\n\n**Explanation:** \n\...
15939,NaN,NaN,Project Evidencia [evidence] - research and ed...,2013,Project Evidencia [Evidence] intends to promot...,"Evidence-based nursing; Databases, bibliograph...",en,**Decision:** Excluded\n\n**Explanation:** \n...
15940,NaN,NaN,Use of Research by Undergraduate Nursing Stude...,2017,Research utilization (RU) is crucial to prepar...,Nursing; Students; Nursing Research; Research ...,en,**Decision:** Included\n\n**Explain:**\n\nThe ...
15941,NaN,NaN,Effects of an Eating Behaviour Modification Pr...,2013,The purpose of this study was to evaluate the ...,Eating behaviour; diabetes; obesity; modificat...,en,Decision: Excluded\n\nExplain: The abstract of...


In [ ]:

# Define a function to extract "Inclusion" or "Exclusion" from the Decision column
def extract_decision(decision_text):
    decision_text = str(decision_text)
    match = re.search(r'\b(Inclusion|Exclusion|Included|Excluded)\b', decision_text, re.IGNORECASE)
    if match:
        return match.group(0).capitalize()  # Capitalize to maintain consistency (e.g., "Excluded")
    return "Unknown"  # Return "Unknown" if no match is found

# Apply the function to the Decision column
processed_df['Decision Type'] = processed_df['Decision'].apply(extract_decision)

# Display the DataFrame with the extracted decision type
print(processed_df[['Decision', 'Decision Type']].head())


                                            Decision Decision Type
0  Excluded\n\nExplain: The title and abstract of...      Excluded
1  **Decision:** Excluded\n\n**Explanation:** \n\...      Excluded
2  **Decision:** Excluded\n\n**Explanation:** \nT...      Excluded
3  Decision: excluded\n\nExplain: The abstract in...      Excluded
4  Excluded\n\nExplain: The abstract of the paper...      Excluded


In [ ]:
filter_data = processed_df[(processed_df['Decision Type'] == 'Included') | (processed_df['Decision Type'] == 'Inclusion')]

# Checking if it is a review study

In [ ]:
ip_port = "132.72.66.154:9982"
url = f"http://{ip_port}/api/generate"


In [ ]:
import json
import pandas as pd
import requests

def extract_full_text(response_text):
    """
    This function extracts and concatenates text from the 'response' field in each JSON line of the response.

    Parameters:
    response_text (str): The raw response text, with each line being a JSON string.

    Returns:
    str: The concatenated text from all 'response' fields.
    """
    full_text = ""  # Variable to store the complete text

    # Iterate over each line in the response
    for line in response_text.text.splitlines():
        try:
            data = json.loads(line)  # Parse each JSON line
            if "response" in data:  # Check if the "response" field exists
                full_text += data["response"]  # Append the text
        except json.JSONDecodeError:
            print("Unable to parse line:", line)

    return full_text

In [ ]:

def check_type(abstract,title):
    prompt = (
        "You will receive a title and an abstract. Your task is to determine the type of review described by the provided information."
        "If the research is a review type (e.g., systematic review, scoping review, meta-analysis), return 0."
        "If the research describes other types of research, return 1."
        "If the type of research cannot be determined from the provided information, return 2. \n\n"
        f"Title: {title}\n"
        f"Abstract: {abstract}\n\n"
        "Decision: {Write the decision here (only number)}"
        "Explain: {Here}"

    )

    payload = {
        "model": "phi4",
        "prompt": prompt,
        "temperature": 0.2,
        "top_p": 0.2
    }

    # Make a POST request
    response = requests.post(url, json=payload)

    # Validate the response
    if response.status_code == 200:
        # decision = response.json().get("choices", [{}])[0].get("text", "").strip()
        decision = extract_full_text(response)
        # print(decision)
        # print("------------------------------------------------")
        return decision #if decision in {"included", "excluded"} else decision
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [ ]:

def check_type(abstract,title):
    prompt = (
        "You will receive a title and an abstract. Your task is to determine the type of review described by the provided information."
        "If the research is a review type (e.g., systematic review, scoping review, meta-analysis), return 0."
        "If the research describes other types of research, return 1."
        "If the type of research cannot be determined from the provided information, return 2. \n\n"
        f"Title: {title}\n"
        f"Abstract: {abstract}\n\n"
        "Write according to the following template:"
        "Decision: {Write the decision here (only number 0,1,2)}"
        "Do not add additional comments."

    )

    payload = {
        "model": "phi4",
        "prompt": prompt,
        "temperature": 0.2,
        "top_p": 0.2
    }

    # Make a POST request
    response = requests.post(url, json=payload)

    # Validate the response
    if response.status_code == 200:
        # decision = response.json().get("choices", [{}])[0].get("text", "").strip()
        decision = extract_full_text(response)
        # print(decision)
        # print("------------------------------------------------")
        return decision #if decision in {"included", "excluded"} else decision
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [ ]:
from tqdm import tqdm

# Define a function to process each row and update the CSV incrementally
def add_decision_column_and_save(df, output_file):
    # Check if the file already exists
    import os
    if os.path.exists(output_file):
        # If the file exists, load it
        processed_df = pd.read_csv(output_file)
        start_index = len(processed_df)  # Start from where we left off
    else:
        # If the file doesn't exist, initialize an empty DataFrame
        processed_df = pd.DataFrame(columns=df.columns.tolist() + ['Type_article'])
        start_index = 0

    # Iterate over rows with tqdm for progress tracking, starting from the last processed row
    for i in tqdm(range(start_index, len(df)), desc="Processing rows"):
        # Extract abstract and title for each row
        abstract = df.iloc[i].Abstract
        title = df.iloc[i].Title

        # Call the check_relevance function
        decision = check_type(abstract, title)

        # Add the current row and decision to the processed DataFrame
        current_row = df.iloc[i].to_dict()
        current_row['Type_article'] = decision
        processed_df = pd.concat([processed_df, pd.DataFrame([current_row])], ignore_index=True)

        # Save the updated DataFrame to the CSV file after each iteration
        processed_df.to_csv(output_file, index=False)

    return processed_df


# File to save the updated DataFrame
output_file = "updated_decisions_incremental_Type_article.csv"

# Call the function with your DataFrame
df_with_decisions = add_decision_column_and_save(filter_data, output_file)

# Display the first few rows of the final DataFrame
print(df_with_decisions.head())


Processing rows:   0%|          | 0/722 [00:00<?, ?it/s]/tmp/ipykernel_3941164/3636865163.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  processed_df = pd.concat([processed_df, pd.DataFrame([current_row])], ignore_index=True)
Processing rows: 100%|██████████| 722/722 [36:33<00:00,  3.04s/it]

    PubMed ID                                 DOI  \
0   7567566.0                                 NaN   
1  17100863.0    10.1111/j.1365-2753.2006.00632.x   
2  12470013.0           10.1016/s0964339702000125   
3  18838895.0  10.1097/01.NND.0000320688.57308.f9   
4  15913267.0                                 NaN   

                                               Title  Year  \
0  Barriers and facilitators of research utilizat...  1995   
1  Research utilization and clinical nurse educat...  2006   
2  Barriers to research utilisation: the clinical...  2002   
3  Nursing faculty mentors as facilitators for ev...  2008   
4  Barriers to implementing research in clinical ...  2005   

                                            Abstract Author Keywords english  \
0  For over 20 years, nursing literature has disc...              []      en   
1  Clinical nurse educators and other linking age...              []      en   
2  The advance of the evidence-based practice (EB...              []

In [7]:
import pandas as pd

df_with_decisions= pd.read_csv('542_with_review_3.csv')

df_with_decisions

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english,Decision,Decision Type,Type_article,Extracted_Number
0,17100863.0,10.1111/j.1365-2753.2006.00632.x,Research utilization and clinical nurse educat...,2006,Clinical nurse educators and other linking age...,[],en,**Included**\n\n**Explanation:**\n- **Inclusio...,Included,**Decision:** 0\n\n**Explanation:** The title ...,0
1,12470013.0,10.1016/s0964339702000125,Barriers to research utilisation: the clinical...,2002,The advance of the evidence-based practice (EB...,[],en,**Decision:** included\n\n**Explain:** \n\nThe...,Included,**Decision:** 1\n\n**Explanation:** The abstra...,1
2,18838895.0,10.1097/01.NND.0000320688.57308.f9,Nursing faculty mentors as facilitators for ev...,2008,Increasing use of evidence-based practice (EBP...,[],en,Included\n\nExplain: \n\nThe abstract of the p...,Included,Decision: 1\n\nExplanation: The abstract descr...,1
3,18710376.0,10.1111/j.1365-2702.2007.02040.x,The spectrum of barriers to and facilitators o...,2008,The focus of the study is the identification o...,[],en,**Decision:** included\n\n**Explanation:**\n\n...,Included,Decision: 1\n\nExplanation: The abstract descr...,1
4,18243707.0,10.1016/j.iccn.2007.12.001,Research activities and perceptions of barrier...,2008,Research utilization is vital to promote evide...,[],en,**Decision:** Included\n\n**Explanation:** \n\...,Included,**Decision:** 1\n\n**Explanation:** The provid...,1
...,...,...,...,...,...,...,...,...,...,...,...
537,NaN,NaN,Building research capacity in the nursing work...,2010,Objectives The Nurse Researcher Project (NRP) ...,Capacity building; cancer care; evidence based...,en,**Included**\n\n**Explanation:** \n\n1. **Incl...,Included,**Decision:** 1\n\n**Explanation:** The abstra...,1
538,NaN,NaN,BARRIERS TO NURSING RESEARCH: A COMPARATIVE ST...,2014,Nurses need to adapt to new models of care bas...,NURSING RESEARCH; COMPREHENSIVE HEALTH CARE; P...,en,Included\n\nExplain:\n\n- Inclusion Criterion ...,Included,**Decision:** 1\n\n**Explanation:** The abstra...,1
539,NaN,NaN,Barriers to research utilization among nurses ...,2011,Aim: The study examines the nurses working at ...,Research utilization; nursing research; barrie...,en,**Decision:** Included\n\n**Explain:** \n\nThe...,Included,**Decision:** 1\n\n**Explanation:** The provid...,1
540,NaN,NaN,RESEARCH UTILIZATION IN NURSING PRACTICE: CLOS...,2012,Background and Significance: The gap between t...,Research Utilization; Nursing Practice; Gap,en,**Included**\n\n**Explanation:** \n\nThe abstr...,Included,Decision: 1\n\nExplanation: The provided title...,1


In [ ]:
from tqdm import tqdm



df_with_decisions["Extracted_Number"] = df_with_decisions["Type_article"].str.extract(r'(?:\*\*Decision:\*\*|Decision:)\s*(\d+)')
# missing_values = df_with_decisions["Extracted_Number"].isnull()
condition = (df_with_decisions['Extracted_Number'] != '0') & (df_with_decisions['Extracted_Number'] != '1')

# Update the 'Type_article' column for rows matching the condition
df_with_decisions.loc[condition, "Type_article"] = df_with_decisions.loc[condition].apply(
    lambda row: check_type(row["Title"], row["Abstract"]), axis=1
)

# df_with_decisions.loc[missing_values, "Type_article"] = df_with_decisions.loc[missing_values].apply(
#     lambda row: check_type(row["Title"], row["Abstract"]), axis=1
# )
# df_with_decisions["Extracted_Number"] = df_with_decisions["Type_article"].str.extract(r'(?:\*\*Decision:\*\*|Decision:)\s*(\d+)')

# Define a regex pattern to extract numbers after '**Decision**:' or 'Decision:'
pattern = r'(?:(?:\*\*Decision\*\*)|Decision):\s*(\d+)\.'
pattern = r"(?:(?:\*\*Decision\*\*)|Decision):\s*(\d+)"

# Apply the regex to the 'Type_article' column where 'Extracted_Number' is missing
missing_values = df_with_decisions["Extracted_Number"].isnull()
df_with_decisions.loc[missing_values, "Extracted_Number"] = (
    df_with_decisions.loc[missing_values, "Type_article"]
    .dropna()
    .str.extract(pattern)[0]
)

# df_with_decisions.loc[missing_values, "Extracted_Number"] = df_with_decisions.loc[missing_values, "Type_article"].str.extract(r'(?:(?:\*\*Decision\*\*)|Decision):\s*(\d+)')








In [7]:
sys = df_with_decisions[(df_with_decisions['Extracted_Number'] != '0') & (df_with_decisions['Extracted_Number'] != '1')]
sys

In [ ]:
df_with_decisions.loc[419,'Extracted_Number'] = '2'
df_with_decisions.loc[421,'Extracted_Number'] = '1'
df_with_decisions.loc[424,'Extracted_Number'] = '2'

In [ ]:
lst_sys = ['systematic review']

df_filtered = df_with_decisions[~df_with_decisions['TITLE'].str.lower().str.contains('|'.join(lst_sys))]

df_with_decisions = df_filtered


In [ ]:
df_with_decisions.to_excel("485_not_systematic_final_4.xlsx")